In [3]:
from langchain_core.documents import Document

In [4]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("../data/LTM - Privacy notice.pdf")
documents = loader.load()

C:\Users\JENNY\AppData\Local\Temp\ipykernel_19508\522364053.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [5]:
from langchain_community.document_loaders import DirectoryLoader

dir_loader = DirectoryLoader(
    "../data",
    glob="*.pdf",
    loader_cls=PyPDFLoader,
    show_progress=False
)

documents = dir_loader.load()
documents

[Document(metadata={'producer': 'calibre 3.23.0 [https://calibre-ebook.com]', 'creator': 'calibre 3.23.0 [https://calibre-ebook.com]', 'creationdate': '2018-06-07T10:08:36+00:00', 'author': 'Jon Erickson', 'keywords': 'COMPUTERS / Security / General', 'title': 'Hacking: The Art of Exploitation, 2nd Edition', 'source': '..\\data\\Hacking - The Art of Exploitation, 2nd Edition by Jon Erickson.pdf', 'total_pages': 622, 'page': 0, 'page_label': '1'}, page_content=''),
 Document(metadata={'producer': 'calibre 3.23.0 [https://calibre-ebook.com]', 'creator': 'calibre 3.23.0 [https://calibre-ebook.com]', 'creationdate': '2018-06-07T10:08:36+00:00', 'author': 'Jon Erickson', 'keywords': 'COMPUTERS / Security / General', 'title': 'Hacking: The Art of Exploitation, 2nd Edition', 'source': '..\\data\\Hacking - The Art of Exploitation, 2nd Edition by Jon Erickson.pdf', 'total_pages': 622, 'page': 1, 'page_label': '2'}, page_content='Hacking:\tThe\tArt\tof\tExploitation,\n2nd\tEdition\nJon\tErickson

In [6]:
type(documents[0])
documents[11]

Document(metadata={'producer': 'calibre 3.23.0 [https://calibre-ebook.com]', 'creator': 'calibre 3.23.0 [https://calibre-ebook.com]', 'creationdate': '2018-06-07T10:08:36+00:00', 'author': 'Jon Erickson', 'keywords': 'COMPUTERS / Security / General', 'title': 'Hacking: The Art of Exploitation, 2nd Edition', 'source': '..\\data\\Hacking - The Art of Exploitation, 2nd Edition by Jon Erickson.pdf', 'total_pages': 622, 'page': 11, 'page_label': '12'}, page_content="past\tto\tthe\tpresent,\tdissecting\tthem\tto\tlearn\thow\tand\twhy\tthey\nwork.\tIncluded\twith\tthis\tbook\tis\ta\tbootable\tLiveCD\tcontaining\nall\tthe\tsource\tcode\tused\therein\tas\twell\tas\ta\tpreconfigured\nLinux\tenvironment.\tExploration\tand\tinnovation\tare\tcritical\tto\nthe\tart\tof\thacking,\tso\tthis\tCD\twill\tlet\tyou\tfollow\talong\tand\nexperiment\ton\tyour\town.\tThe\tonly\trequirement\tis\tan\t\nx\n86\nprocessor,\twhich\tis\tused\tby\tall\tMicrosoft\tWindows\tmachines\nand\tthe\tnewer\tMacintosh\tcomputer

In [7]:
from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [8]:
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            all_documents.extend(documents)
            print(f" Loaded {len(documents)} pages")

        except Exception as e:
            print(f" Error: {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

all_pdf_docs = process_all_pdfs("../data")


Found 2 PDF files to process

Processing: Hacking - The Art of Exploitation, 2nd Edition by Jon Erickson.pdf
 Loaded 622 pages

Processing: LTM - Privacy notice.pdf
 Loaded 23 pages

Total documents loaded: 645


In [9]:
all_pdf_docs

[Document(metadata={'producer': 'calibre 3.23.0 [https://calibre-ebook.com]', 'creator': 'calibre 3.23.0 [https://calibre-ebook.com]', 'creationdate': '2018-06-07T10:08:36+00:00', 'author': 'Jon Erickson', 'keywords': 'COMPUTERS / Security / General', 'title': 'Hacking: The Art of Exploitation, 2nd Edition', 'source': '..\\data\\Hacking - The Art of Exploitation, 2nd Edition by Jon Erickson.pdf', 'total_pages': 622, 'page': 0, 'page_label': '1', 'source_file': 'Hacking - The Art of Exploitation, 2nd Edition by Jon Erickson.pdf', 'file_type': 'pdf'}, page_content=''),
 Document(metadata={'producer': 'calibre 3.23.0 [https://calibre-ebook.com]', 'creator': 'calibre 3.23.0 [https://calibre-ebook.com]', 'creationdate': '2018-06-07T10:08:36+00:00', 'author': 'Jon Erickson', 'keywords': 'COMPUTERS / Security / General', 'title': 'Hacking: The Art of Exploitation, 2nd Edition', 'source': '..\\data\\Hacking - The Art of Exploitation, 2nd Edition by Jon Erickson.pdf', 'total_pages': 622, 'page'

In [10]:
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)}")

    if split_docs:
        print(f"\nContent: {split_docs[1].page_content[:200]}")
        print(f"Metadata: {split_docs[1].metadata}")
    return split_docs


In [11]:
chunks = split_documents(all_pdf_docs)
chunks

Split 645 documents into 1487

Content: HACKING:	THE	ART	OF	EXPLOITATION,	2ND
EDITION.
Copyright	
©	2008	by	Jon	Erickson.
All	rights	reserved.	No	part	of	this	work	may	be	reproduced	or
transmitted	in	any	form	or	by	any	means,	electronic	or

Metadata: {'producer': 'calibre 3.23.0 [https://calibre-ebook.com]', 'creator': 'calibre 3.23.0 [https://calibre-ebook.com]', 'creationdate': '2018-06-07T10:08:36+00:00', 'author': 'Jon Erickson', 'keywords': 'COMPUTERS / Security / General', 'title': 'Hacking: The Art of Exploitation, 2nd Edition', 'source': '..\\data\\Hacking - The Art of Exploitation, 2nd Edition by Jon Erickson.pdf', 'total_pages': 622, 'page': 2, 'page_label': '3', 'source_file': 'Hacking - The Art of Exploitation, 2nd Edition by Jon Erickson.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'calibre 3.23.0 [https://calibre-ebook.com]', 'creator': 'calibre 3.23.0 [https://calibre-ebook.com]', 'creationdate': '2018-06-07T10:08:36+00:00', 'author': 'Jon Erickson', 'keywords': 'COMPUTERS / Security / General', 'title': 'Hacking: The Art of Exploitation, 2nd Edition', 'source': '..\\data\\Hacking - The Art of Exploitation, 2nd Edition by Jon Erickson.pdf', 'total_pages': 622, 'page': 1, 'page_label': '2', 'source_file': 'Hacking - The Art of Exploitation, 2nd Edition by Jon Erickson.pdf', 'file_type': 'pdf'}, page_content='Hacking:\tThe\tArt\tof\tExploitation,\n2nd\tEdition\nJon\tErickson\nEditor\nWilliam\tPollock\nCopyright\t©\t2010'),
 Document(metadata={'producer': 'calibre 3.23.0 [https://calibre-ebook.com]', 'creator': 'calibre 3.23.0 [https://calibre-ebook.com]', 'creationdate': '2018-06-07T10:08:36+00:00', 'author': 'Jon Erickson', 'keywords': 'COMPUTERS / Security / General', 'title': 'Hacking: The Art of Exploitation, 2nd Edition', 'sou

In [12]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity


In [13]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager

        Args:
             model_name: HuggingFace model name for sentence embeddings
        """

        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the sentenceTransformer model"""

        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise


    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        Args:
            texts: list of text strings to embed

        Return:
             numpy array of embeddings with shape (len(texts), embedding_dim)
            """
        if not self.model:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 679.13it/s]


Model loaded successfully Embedding dimension: 384


In [14]:
import os
class VectorStore:
    """Manages document embeddings in a chromaDB vector database"""

    def __init__(self, collection_name: str = "pdf_docs", persist_directory: str = "../data/vector_store"):
        """Initialize the vector store
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection."""
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"desscription": "PDF document embeddings for RAG"}
            )
        except Exception as e:
            print(f"Error initializing v s : {e}")

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        Args:
            documents: List of LangChain documents
            embeddings: corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match no of embeddings")
        print(f"Adding {len(documents)}documents to vector store")

        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            documents_text.append(doc.page_content)
            embeddings_list.append(embedding.tolist())

        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error {e}")
            raise

vector_store=VectorStore()
vector_store

In [15]:
chunks
texts=[doc.page_content for doc in chunks]
texts

embeddings=embedding_manager.generate_embeddings(texts)
embeddings
vector_store.add_documents(chunks, embeddings)

Generating embeddings for 1487 texts


Batches: 100%|██████████| 47/47 [01:45<00:00,  2.25s/it]


Generated embeddings with shape: (1487, 384)
Adding 1487documents to vector store
Successfully added 1487 documents to vector store
Total documents in collection: 2974


In [16]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """

        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query

        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold

        Return:
              List of dictionaries containing retrieved documents and metadate
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            retrieved_docs=[]

            if results['documents'] and results["documents"][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                        print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")

            else:
                print("No documents found")

            return retrieved_docs

        except Exception as e:
          print(f"Error during retreival: {e}")
          return []

rag_retriever = RAGRetriever(vector_store, embedding_manager)

In [17]:
rag_retriever.retrieve("The	current	laws restricting cryptography and cryptographic research further blur the line between hackers and crackers.")

Retrieving documents for query: 'The	current	laws restricting cryptography and cryptographic research further blur the line between hackers and crackers.'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00,  9.04it/s]

Generated embeddings with shape: (1, 384)


Retrieved 1 documents (after filtering)
Retrieved 2 documents (after filtering)
Retrieved 3 documents (after filtering)
Retrieved 4 documents (after filtering)
Retrieved 5 documents (after filtering)


[{'id': 'doc_11dfbdba_18',
  'content': "technically\tillegal\tto\treverse\tengineer\tor\teven\tdiscuss\tPig\tLatin\nif\tit\twere\tused\tas\tan\tindustry\tconsumer\tcontrol.\tWho\tare\tthe\nhackers\tand\twho\tare\tthe\tcrackers\tnow?\tWhen\tlaws\tseem\tto\ninterfere\twith\tfree\tspeech,\tdo\tthe\tgood\tguys\twho\tspeak\ttheir\nminds\tsuddenly\tbecome\tbad?\tI\tbelieve\tthat\tthe\tspirit\tof\tthe\nhacker\ttranscends\tgovernmental\tlaws,\tas\topposed\tto\tbeing\ndefined\tby\tthem.\nThe\tsciences\tof\tnuclear\tphysics\tand\tbiochemistry\tcan\tbe\tused\nto\tkill,\tyet\tthey\talso\tprovide\tus\twith\tsignificant\tscientific\nadvancement\tand\tmodern\tmedicine.\tThere's\tnothing\tgood\tor\nbad\tabout\tknowledge\titself;\tmorality\tlies\tin\tthe\tapplication\tof\nknowledge.\tEven\tif\twe\twanted\tto,\twe\tcouldn't\tsuppress\tthe\nknowledge\tof\thow\tto\tconvert\tmatter\tinto\tenergy\tor\tstop\tthe\ncontinued\ttechnological\tprogress\tof\tsociety.\tIn\tthe\tsame\tway,\nthe\thacker\tspirit\tcan

In [20]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv

llm = ChatGroq(model_name = "openai/gpt-oss-120b", temperature=0.1, max_tokens=1024)

def rag_simple(query, retriever, llm, top_k=3):
    results = retriever.retrieve(query, top_k=top_k)
    context = "\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant information found to answer the question"

    prompt=f"""Use the following context to answer the questons consicely.
        Context:
        {context}

        Question: {query}

        Answer:"""

    response=llm.invoke([prompt])
    return response.content

In [21]:
answer=rag_simple("RC4	Stream	Cipher ?", rag_retriever, llm)
answer

Retrieving documents for query: 'RC4	Stream	Cipher ?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 17.41it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)
Retrieved 2 documents (after filtering)
Retrieved 3 documents (after filtering)


'RC4 is a symmetric stream cipher that generates a pseudo‑random keystream by maintaining a 256‑byte state array (S‑box). It works in two phases:\n\n* **Key‑Scheduling Algorithm (KSA)** – the secret key (often an IV\u202f+\u202fkey) is mixed into the S‑box to produce an initial permutation.\n* **Pseudo‑Random Generation Algorithm (PRGA)** – the S‑box is continuously permuted, and bytes from it are extracted to form the keystream, which is XOR‑ed with the plaintext (or ciphertext).\n\nThe algorithm is simple, fast, and uses only byte‑wise operations on the 256‑element S‑box. It was widely used (e.g., in WEP) but is now considered insecure due to several biases in its keystream.'

In [44]:
def rag_advanced(
    query: str,
    retriever,
    llm,
    top_k: int = 3,
    score_threshold: float = 0.3
):
    """
    Advanced RAG pipeline.

    Steps:
        1. Retrieve relevant documents
        2. Filter using similarity score
        3. Build context with metadata
        4. Generate grounded answer using LLM
        5. Return answer with sources
    """

    print(f"\nQuery: {query}")
    print(f"Top K: {top_k}")
    print(f"Score threshold: {score_threshold}")

    # --------------------------------------------------
    # 1. Retrieve documents
    # --------------------------------------------------

    results = retriever.retrieve(
        query=query,
        top_k=top_k,
        score_threshold=score_threshold
    )

    if not results:
        return {
            "answer": "I could not find relevant information in the provided documents.",
            "sources": [],
            "retrieved_documents": 0
        }

    # --------------------------------------------------
    # 2. Build context
    # --------------------------------------------------

    context_parts = []

    for i, result in enumerate(results):

        metadata = result.get("metadata", {})

        source = metadata.get(
            "source",
            metadata.get("file_path", "Unknown")
        )

        page = metadata.get(
            "page",
            metadata.get("page_number", "Unknown")
        )

        similarity_score = result.get(
            "similarity_score",
            0.0
        )

        context_parts.append(
            f"""
--- Document {i + 1} ---
Source: {source}
Page: {page}
Similarity Score: {similarity_score:.4f}

Content:
{result['content']}
"""
        )

    context = "\n".join(context_parts)

    # --------------------------------------------------
    # 3. Create grounded RAG prompt
    # --------------------------------------------------

    prompt = f"""
You are a helpful question-answering assistant.

Answer the user's question using ONLY the information
provided in the context below.

Rules:
1. Do not use outside knowledge.
2. Do not invent or assume information.
3. If the answer cannot be found in the context,
   clearly say that the information is not available.
4. Give a concise but complete answer.
5. When possible, mention the source and page.
6. If multiple documents contain relevant information,
   combine them carefully.
7. Distinguish between facts explicitly stated in the
   documents and information that is not available.

Context:
{context}

User Question:
{query}

Answer:
"""

    # --------------------------------------------------
    # 4. Generate answer
    # --------------------------------------------------

    response = llm.invoke(prompt)

    answer = response.content

    # --------------------------------------------------
    # 5. Prepare sources
    # --------------------------------------------------

    sources = []

    for result in results:

        metadata = result.get("metadata", {})

        sources.append({
            "id": result.get("id"),
            "source": metadata.get("source"),
            "page": metadata.get("page"),
            "similarity_score": result.get("similarity_score")
        })

    # --------------------------------------------------
    # 6. Return complete RAG response
    # --------------------------------------------------

    return {
        "answer": answer,
        "sources": sources,
        "retrieved_documents": len(results),
        "similarity_score": similarity_score
    }

In [47]:
response = rag_advanced("Brute forcing will always be a possible attack on any computationally secure cryptosystem", rag_retriever, llm,top_k=3,score_threshold = 0.1)
print(response['answer'])
print(response)


Query: Brute forcing will always be a possible attack on any computationally secure cryptosystem
Top K: 3
Score threshold: 0.1
Retrieving documents for query: 'Brute forcing will always be a possible attack on any computationally secure cryptosystem'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts


Batches: 100%|██████████| 1/1 [00:00<00:00, 28.02it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)
Retrieved 2 documents (after filtering)
Retrieved 3 documents (after filtering)


Yes. For a **computationally secure** cryptosystem, brute‑force (exhaustive‑key) search is theoretically possible, but the time required to succeed is astronomically large—often measured in “tens of thousands of years even with a vast array of computational resources”【Document 3, p. 539】. (By contrast, an **unconditionally secure** system cannot be broken even by exhaustive key search, regardless of resources【Document 1/2, p. 536】.)
{'answer': 'Yes.\u202fFor a **computationally secure** cryptosystem, brute‑force (exhaustive‑key) search is theoretically possible, but the time required to succeed is astronomically large—often measured in “tens of thousands of years even with a vast array of computational resources”【Document\u202f3, p.\u202f539】. (By contrast, an **unconditionally secure** system cannot be broken even by exhaustive key search, regardless of resources【Document\u202f1/2, p.\u202f536】.)', 'sources': [{'id': 'doc_3cc01f5b_1204', 'source': '..\\data\\Hacking - The Art of Explo

In [38]:
query_embedding = embedding_manager.generate_embeddings(
    ["Offline Brute-Force Attacks"]
)[0]

results = vector_store.collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=10
)

print("IDs:")
print(results["ids"])

print("\nDistances:")
print(results["distances"])

print("\nDocuments:")
for i, doc in enumerate(results["documents"][0]):
    print("=" * 80)
    print("RESULT:", i + 1)
    print(doc[:1000])

Generating embeddings for 1 texts


Batches: 100%|██████████| 1/1 [00:00<00:00, 57.00it/s]

Generated embeddings with shape: (1, 384)
IDs:
[['doc_1b7268b2_1301', 'doc_a87010db_1301', 'doc_a8f0eb48_1335', 'doc_655574aa_1335', 'doc_581c4c75_5', 'doc_bc0e473f_5', 'doc_7261f84c_760', 'doc_dee2bcee_760', 'doc_d9a7bade_20', 'doc_e98f30d6_20']]

Distances:
[[1.0374222993850708, 1.0374222993850708, 1.0809807777404785, 1.0809807777404785, 1.0847411155700684, 1.0847411155700684, 1.0969607830047607, 1.0969607830047607, 1.1205127239227295, 1.1205127239227295]]

Documents:
RESULT: 1
guesses:	0		time:	0:00:00:03	6%	(2)		c/s:	5489		trying:	exports
guesses:	0		time:	0:00:00:05	10%	(2)		c/s:	5561		trying:	catcat
guesses:	0		time:	0:00:00:09	20%	(2)		c/s:	5514		trying:	dilbert!
guesses:	0		time:	0:00:00:10	22%	(2)		c/s:	5513		trying:	redrum3
testing7									(jose)
guesses:	1		time:	0:00:00:14	44%	(2)		c/s:	5539		trying:	KnightKnight
guesses:	1		time:	0:00:00:17	59%	(2)		c/s:	5572		trying:	Gofish!	
Session	aborted
In	this	output,	the	account	jose	is	shown	to	have	the	password
of	
testing7
.
Ha